In [ ]:
!pip install -q transformers datasets peft trl bitsandbytes accelerate scikit-learn
from huggingface_hub import login
login()   # paste your HF token
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch, json, os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE = "meta-llama/Llama-3.2-3B-Instruct"
DRIVE = "/content/drive/MyDrive/finsent"
RUN = "r16_lr2e-4"
ADAPTER = f"{DRIVE}/adapter_{RUN}"
assert os.path.isdir(ADAPTER), "adapter folder not found"

tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
base = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb,
                                            device_map={"": 0}, dtype=torch.float16)
model = PeftModel.from_pretrained(base, ADAPTER)
model.eval()
print("loaded adapter:", ADAPTER)

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

loaded adapter: /content/drive/MyDrive/finsent/adapter_r16_lr2e-4


In [ ]:
import torch
from datasets import load_dataset
from tqdm.auto import tqdm

ds = load_dataset("zeroshot/twitter-financial-news-sentiment")
feat = ds["train"].features["label"]
label_words = [n.capitalize() for n in feat.names]
print("Labels:", label_words)
assert label_words == ["Bearish", "Bullish", "Neutral"]

PROMPT_DATE = "01 Jan 2026"
def build_prompt(text):
    msgs = [
        {"role": "system", "content": "You are a financial sentiment classifier."},
        {"role": "user", "content": f"Classify the sentiment of this finance tweet as exactly one of: {', '.join(label_words)}.\n\nTweet: {text}\n\nAnswer with one word."}
    ]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, date_string=PROMPT_DATE)

print("Sample prompt snippet:")
print(build_prompt("test")[:200])

tok.padding_side = "left"

# We encode the label words with a leading space (or as they appear after the template)
# to ensure we capture the actual token the model will generate next.
label_token_ids = []
for w in label_words:
    # Encode with a space prepended since it follows the generation prompt delimiter
    ids = tok.encode(" " + w, add_special_tokens=False)
    label_token_ids.append(ids[0])

print("Label token IDs:", label_token_ids)
print("Decoded:", [tok.decode([t]) for t in label_token_ids])
assert len(set(label_token_ids)) == 3, "Label tokens are not unique!"

@torch.no_grad()
def predict(texts, bs=16):
    preds = []
    for i in tqdm(range(0, len(texts), bs)):
        enc = tok([build_prompt(t) for t in texts[i:i+bs]], return_tensors="pt",
                  padding=True, add_special_tokens=False).to(model.device)
        outputs = model(**enc)
        # Extract logits of the very last generated token
        logits = outputs.logits[:, -1, :]
        # Restrict to our class label token IDs and find the max index
        preds += logits[:, label_token_ids].argmax(-1).cpu().tolist()
    return preds

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Labels: ['Bearish', 'Bullish', 'Neutral']
Sample prompt snippet:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 01 Jan 2026

You are a financial sentiment classifier.<|eot_id|><|start_header_id|>user<|
Label token IDs: [24941, 22353, 59794]
Decoded: [' Bear', ' Bull', ' Neutral']


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

val_texts = ds["validation"]["text"]
val_y = ds["validation"]["label"]

def report(name, preds):
    out = {"name": name, "n": len(val_y),
           "accuracy": accuracy_score(val_y, preds),
           "macro_f1": f1_score(val_y, preds, average="macro"),
           "confusion_matrix": confusion_matrix(val_y, preds).tolist(),
           "per_class": classification_report(val_y, preds, target_names=label_words, output_dict=True)}
    print(name, "| acc", round(out["accuracy"], 4), "| macro-F1", round(out["macro_f1"], 4))
    print(out["confusion_matrix"])
    print(classification_report(val_y, preds, target_names=label_words))
    return out

# 1) fine-tuned
ft_preds = predict(val_texts)
ft = report("finetuned_" + RUN, ft_preds)

# 2) base = same loaded model with adapter disabled (expect macro-F1 near 0.557)
with model.disable_adapter():
    base_preds = predict(val_texts)
bs_ = report("base_same_loaded_model", base_preds)

json.dump({**ft, "preds": ft_preds}, open(f"{DRIVE}/eval_finetuned_{RUN}.json", "w"))
json.dump({**bs_, "preds": base_preds}, open(f"{DRIVE}/eval_base_{RUN}.json", "w"))
print("saved to Drive")
print("Pre-registered bar: fine-tuned macro-F1 > 0.697 (TF-IDF + LogReg)")

  0%|          | 0/150 [00:00<?, ?it/s]

finetuned_r16_lr2e-4 | acc 0.902 | macro-F1 0.8734
[[290, 0, 57], [1, 346, 128], [35, 13, 1518]]
              precision    recall  f1-score   support

     Bearish       0.89      0.84      0.86       347
     Bullish       0.96      0.73      0.83       475
     Neutral       0.89      0.97      0.93      1566

    accuracy                           0.90      2388
   macro avg       0.91      0.84      0.87      2388
weighted avg       0.91      0.90      0.90      2388



  0%|          | 0/150 [00:00<?, ?it/s]

base_same_loaded_model | acc 0.6059 | macro-F1 0.6061
[[327, 7, 13], [24, 433, 18], [400, 479, 687]]
              precision    recall  f1-score   support

     Bearish       0.44      0.94      0.60       347
     Bullish       0.47      0.91      0.62       475
     Neutral       0.96      0.44      0.60      1566

    accuracy                           0.61      2388
   macro avg       0.62      0.76      0.61      2388
weighted avg       0.78      0.61      0.60      2388

saved to Drive
Pre-registered bar: fine-tuned macro-F1 > 0.697 (TF-IDF + LogReg)


In [ ]:
bnb2 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                          bnb_4bit_compute_dtype=torch.float16)
m2 = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb2, device_map="auto")
m2.eval()
model_ft = model          # keep the adapter model
model = m2                # predict() uses the global `model`
p2 = predict(val_texts)
model = model_ft
_ = report("base_baseline_config_no_dq", p2)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s]

base_baseline_config_no_dq | acc 0.6022 | macro-F1 0.6028
[[328, 7, 12], [25, 432, 18], [407, 481, 678]]
              precision    recall  f1-score   support

     Bearish       0.43      0.95      0.59       347
     Bullish       0.47      0.91      0.62       475
     Neutral       0.96      0.43      0.60      1566

    accuracy                           0.60      2388
   macro avg       0.62      0.76      0.60      2388
weighted avg       0.78      0.60      0.60      2388



In [ ]:
import transformers, peft, bitsandbytes
print(transformers.__version__, peft.__version__, bitsandbytes.__version__, torch.__version__)
print("default-load dtype:", m2.get_input_embeddings().weight.dtype)

del m2; torch.cuda.empty_cache()
m3 = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb2,
                                          device_map="auto", dtype=torch.float32)
m3.eval()
model_ft = model
model = m3
p3 = predict(val_texts)
model = model_ft
_ = report("base_fp32_nonquant_layers", p3)

5.22.1 0.21.2 0.50.2 2.14.1+cu130
default-load dtype: torch.bfloat16


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s]

base_fp32_nonquant_layers | acc 0.6085 | macro-F1 0.6086
[[328, 7, 12], [23, 434, 18], [399, 476, 691]]
              precision    recall  f1-score   support

     Bearish       0.44      0.95      0.60       347
     Bullish       0.47      0.91      0.62       475
     Neutral       0.96      0.44      0.60      1566

    accuracy                           0.61      2388
   macro avg       0.62      0.77      0.61      2388
weighted avg       0.79      0.61      0.61      2388

